# Step 4: Video Segmentation and Upload

This notebook cuts the original videos into chunks based on the semantic plan from Step 3, optimizes them for web playback, and optionally uploads everything to an S3-compatible bucket.

**What it does:**
1. Reads each `*_plan.json` from Step 3
2. Cuts the original video at the specified timestamps using FFmpeg
3. Extracts the corresponding transcript sentences for each chunk
4. Creates a structured folder per chunk with `video.mp4` + `data.json`
5. (Optional) Uploads the full structure to an S3-compatible bucket
6. (Optional) Creates a zip archive of the JSON/transcript data (without videos)

**Input:** Original videos, transcript JSONs, and plan JSONs from previous steps  
**Output:** Structured folders ready for a web-based course player

```
Final_Chunks/
  Day_1/
    Video_Title/
      Video_Title.json          <- full original transcript
      01_introduction/
        video.mp4               <- web-optimized clip (720p)
        data.json               <- chunk title, summary, sentences
      02_core_concepts/
        video.mp4
        data.json
      ...
```

---
## Prerequisites
- Original video files and transcript JSONs from Steps 1-2
- Plan JSONs from Step 3 (`03_semantic_chunking.ipynb`)
- (Optional) S3-compatible bucket credentials for upload


## Configuration

In [ ]:
import os
import json
import glob
import subprocess
import re
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ===============================================
# CONFIGURATION - Edit these paths
# ===============================================
TRANSCRIPTS_FOLDER = "/content/drive/MyDrive/your_transcripts/Day_1"
VIDEOS_FOLDER = "/content/drive/MyDrive/your_videos/Day_1"
PLANS_FOLDER = "/content/drive/MyDrive/your_structure/Day_1"
FINAL_OUTPUT_FOLDER = "/content/drive/MyDrive/your_final_chunks/Day_1"

# Video encoding settings
VIDEO_HEIGHT = 720     # Output resolution height (720p)
VIDEO_CRF = 28         # Quality (lower = better quality, 18-28 is typical)
# ===============================================

os.makedirs(FINAL_OUTPUT_FOLDER, exist_ok=True)
print("Configuration loaded.")


## Helper Functions

In [ ]:
def slugify(text):
    """Create a folder-safe name from a chunk title."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')

def format_time(ms):
    """Convert milliseconds to HH:MM:SS.mmm format for FFmpeg."""
    s, ms_rem = divmod(ms, 1000)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d}.{int(ms_rem):03d}"


## Cut Videos and Build Folder Structure

For each video, reads the plan and:
- Creates a subfolder per chunk
- Cuts the video segment with FFmpeg (720p, web-optimized)
- Extracts the matching transcript sentences into `data.json`


In [ ]:
plan_files = sorted(glob.glob(os.path.join(PLANS_FOLDER, "*_plan.json")))
print(f"Found {len(plan_files)} plan(s) to process.\n")

for plan_path in plan_files:
    base_name = os.path.basename(plan_path).replace("_plan.json", "")
    video_path = os.path.join(VIDEOS_FOLDER, f"{base_name}.mp4")
    json_path = os.path.join(TRANSCRIPTS_FOLDER, f"{base_name}.json")

    if not os.path.exists(video_path) or not os.path.exists(json_path):
        print(f"[skip] Missing video or transcript for: {base_name}")
        continue

    # Load data
    with open(json_path, 'r', encoding='utf-8') as f:
        orig_data = json.load(f)
    with open(plan_path, 'r', encoding='utf-8') as f:
        plan = json.load(f)

    # Create main folder for this video
    video_output_dir = os.path.join(FINAL_OUTPUT_FOLDER, base_name)
    os.makedirs(video_output_dir, exist_ok=True)

    # Copy full transcript alongside chunks
    shutil.copy(json_path, os.path.join(video_output_dir, f"{base_name}.json"))

    # Find the sentences list in the transcript JSON
    sentences = []
    if 'transcripts' in orig_data and orig_data['transcripts']:
        sentences = orig_data['transcripts'][0].get('sentences', [])
    elif 'sentences' in orig_data:
        sentences = orig_data['sentences']

    print(f"Processing: {base_name} ({len(plan)} chunks)")

    for idx, chunk in enumerate(plan):
        # Create chunk subfolder
        folder_name = f"{idx+1:02d}_{slugify(chunk['title'])}"
        chunk_dir = os.path.join(video_output_dir, folder_name)
        os.makedirs(chunk_dir, exist_ok=True)

        # Extract matching sentences for this chunk
        sliced = [
            s for s in sentences
            if s.get('sentence_id', -1) >= chunk['start_sentence_id']
            and s.get('sentence_id', -1) <= chunk['end_sentence_id']
        ]

        # Save chunk metadata
        chunk_data = {
            "title": chunk['title'],
            "summary": chunk['summary'],
            "audio_info": orig_data.get('audio_info', {}),
            "sentences": sliced
        }
        with open(os.path.join(chunk_dir, "data.json"), 'w', encoding='utf-8') as f:
            json.dump(chunk_data, f, indent=4, ensure_ascii=False)

        # Cut video with FFmpeg
        start_str = format_time(chunk['begin_time_ms'])
        duration_str = format_time(chunk['end_time_ms'] - chunk['begin_time_ms'])

        subprocess.run([
            'ffmpeg', '-y',
            '-ss', start_str,
            '-i', video_path,
            '-t', duration_str,
            '-vf', f'scale=-1:{VIDEO_HEIGHT}',
            '-crf', str(VIDEO_CRF),
            '-movflags', '+faststart',
            os.path.join(chunk_dir, "video.mp4")
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        print(f"  [{idx+1}/{len(plan)}] {folder_name}")

    print(f"Done: {base_name}\n")

print("All videos segmented.")


## (Optional) Upload to S3-Compatible Bucket

Upload the entire folder structure to an S3-compatible storage service (AWS S3, Railway, Cloudflare R2, MinIO, etc.).

**Skip this cell** if you don't need cloud upload.


In [ ]:
# Uncomment and configure to enable upload
# !pip install boto3

# import boto3
# from pathlib import Path
# from botocore.client import Config
#
# # ===============================================
# # S3 UPLOAD CONFIGURATION
# # ===============================================
# S3_BUCKET_NAME = ""
# S3_ENDPOINT = ""            # e.g., "https://t3.storageapi.dev" for Railway
# S3_ACCESS_KEY_ID = ""
# S3_SECRET_ACCESS_KEY = ""
# S3_REGION = "auto"
# S3_PREFIX = "final_chunks"  # Prefix path inside the bucket
# # ===============================================
#
# s3 = boto3.client(
#     's3',
#     region_name=S3_REGION,
#     endpoint_url=S3_ENDPOINT,
#     aws_access_key_id=S3_ACCESS_KEY_ID,
#     aws_secret_access_key=S3_SECRET_ACCESS_KEY,
#     config=Config(signature_version='s3v4'),
# )
#
# day_name = os.path.basename(FINAL_OUTPUT_FOLDER)
# for file_path in Path(FINAL_OUTPUT_FOLDER).rglob('*'):
#     if file_path.is_file():
#         relative = file_path.relative_to(FINAL_OUTPUT_FOLDER)
#         s3_key = os.path.join(S3_PREFIX, day_name, str(relative))
#         s3.upload_file(str(file_path), S3_BUCKET_NAME, s3_key)
#         print(f"  Uploaded: {s3_key}")
#
# print("Upload complete.")


## (Optional) Export JSON-Only Archive

Creates a zip of the folder structure **without** video files. Useful for backing up transcripts and metadata.


In [ ]:
parent_dir = os.path.dirname(FINAL_OUTPUT_FOLDER)
day_name = os.path.basename(FINAL_OUTPUT_FOLDER)
zip_path = os.path.join(parent_dir, f"{day_name}_transcripts_only.zip")

!cd "{parent_dir}" && zip -r "{zip_path}" "{day_name}" -x "*.mp4"

print(f"Archive saved: {zip_path}")
